In [1]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
import os
import numpy as np
from tensorflow.keras import layers
from IPython.display import Audio
import wave
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

2024-12-22 16:19:17.273433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1734902357.284591   51527 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734902357.287886   51527 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-22 16:19:17.300742: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# %load_ext tensorboard

In [3]:
df = pd.read_csv("../data_processing/sep28k-mfcc.csv")

In [4]:
df = df[df['NaturalPause'] == 0]
df = df[df['Interjection'] == 0]
df = df[df['Prolongation'] == 0]
df = df[df['SoundRep'] == 0]
df = df[df['Block'] == 0]
df.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,...,29,30,31,32,33,34,35,36,37,38
2,HeStutters,0,2,34809760,34857760,0,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
4,HeStutters,0,4,35721920,35769920,0,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
6,HeStutters,0,6,37251200,37299200,0,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
9,HeStutters,0,9,41417440,41465440,0,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
11,HeStutters,0,11,42928800,42976800,0,0,0,0,0,...,-3.798326,-6.705790,-1.544637,-6.356187,-0.987506,-3.107216,-1.422836,-3.615824,-1.171095,-1.937332


In [5]:
df = df.reset_index()
df.head()

,index,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,...,29,30,31,32,33,34,35,36,37,38
0,2,HeStutters,0,2,34809760,34857760,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
1,4,HeStutters,0,4,35721920,35769920,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
2,6,HeStutters,0,6,37251200,37299200,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
3,9,HeStutters,0,9,41417440,41465440,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
4,11,HeStutters,0,11,42928800,42976800,0,0,0,0,...,-3.798326,-6.705790,-1.544637,-6.356187,-0.987506,-3.107216,-1.422836,-3.615824,-1.171095,-1.937332


In [6]:
df = df.drop(columns=['index'])
df.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,...,29,30,31,32,33,34,35,36,37,38
0,HeStutters,0,2,34809760,34857760,0,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
1,HeStutters,0,4,35721920,35769920,0,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
2,HeStutters,0,6,37251200,37299200,0,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
3,HeStutters,0,9,41417440,41465440,0,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
4,HeStutters,0,11,42928800,42976800,0,0,0,0,0,...,-3.798326,-6.705790,-1.544637,-6.356187,-0.987506,-3.107216,-1.422836,-3.615824,-1.171095,-1.937332


In [7]:
df.to_csv("wordrep.csv",index=False)

In [8]:
import os

def list_files(directory):
    return [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

# Replace 'directory_path' with the path to your directory
directory_path = '/home/alien/Git/DATA/mel_spects_wordrep'
# names_list = list_files(directory_path)
names_list = pd.read_csv("wordrep.csv")['Name'].values.tolist()
# full_names_list = ["/home/alien/Git/DATA/mfcc_images/" + img + ".jpg" for img in names_list]
full_names_list = []
for img in names_list:
    corresponding_sound = df.loc[df['Name'] == img, 'WordRep'].values[0]
    if corresponding_sound == 0:
        full_names_list.append("/home/alien/Git/DATA/mel_spects_wordrep/" + img + "_fluent.jpg")
    if corresponding_sound >= 1:
        full_names_list.append("/home/alien/Git/DATA/mel_spects_wordrep/" + img + "_stutter.jpg")



# Replace 'directory_path' with the path to your directory
directory_path = '/home/alien/Git/DATA/mel_specaugment_wordrep'
# names_list = list_files(directory_path)
names_list = pd.read_csv("wordrep.csv")['Name'].values.tolist()
# full_names_list = ["/home/alien/Git/DATA/mfcc_images/" + img + ".jpg" for img in names_list]
for img in names_list:
    corresponding_sound = df.loc[df['Name'] == img, 'WordRep'].values[0]
    if corresponding_sound == 0:
        full_names_list.append("/home/alien/Git/DATA/mel_specaugment_wordrep/" + img + "_fluent.jpg")
    if corresponding_sound >= 1:
        full_names_list.append("/home/alien/Git/DATA/mel_specaugment_wordrep/" + img + "_stutter.jpg")

print(full_names_list[30])
print(full_names_list[-30])

/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_2_141_fluent.jpg
/home/alien/Git/DATA/mel_specaugment_wordrep/WomenWhoStutter_104_22_fluent.jpg


In [9]:
def load_all(imagefile_list):
    data = []
    labels = []

    for imagefile in imagefile_list:
        print(imagefile)
        image = cv2.imread(imagefile)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        data.append(image)

        if "fluent" in imagefile:
            labels.append(0)
        else:
            labels.append(1)

    labels = np.array(labels)
    data = np.array(data)

    return data, labels

In [10]:
X, y = load_all(full_names_list)

/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_2_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_4_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_6_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_9_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_11_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_12_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_15_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_16_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_20_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_22_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_24_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_26_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_29_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_32_fluent.jpg
/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_34_fluent.jpg

In [11]:
print(X[0])
print(y[0])

[[[ 68  33  60]
  [ 96  28  68]
  [ 89  27  67]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 [[114  57  88]
  [166  57 117]
  [157  55 112]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 [[140  60  94]
  [190  55 118]
  [198  65 126]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 ...

 [[126  65 103]
  [194  56 123]
  [189  69 128]
  ...
  [ 29  13  72]
  [ 18   5  59]
  [ 21   6  57]]

 [[ 90  48 102]
  [145  31 116]
  [141  39 122]
  ...
  [ 26  18  68]
  [ 24  17  63]
  [ 27  19  61]]

 [[ 86  33  88]
  [122  36 120]
  [134  36 131]
  ...
  [ 23  14  63]
  [ 27  18  70]
  [ 26  14  66]]]
0


In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=88)

In [13]:
print("x_train shape:", X_train.shape)
print("x_test shape:", X_test.shape)
print('y_train shape:', y_train.shape)
print("y_test shape:", y_test.shape)

x_train shape: (3364, 224, 224, 3)
x_test shape: (842, 224, 224, 3)
y_train shape: (3364,)
y_test shape: (842,)


In [14]:
print(len(X_train), len(X_test), len(y_train), len(y_test))

3364 842 3364 842


In [15]:
from collections import Counter
print(Counter(y_train))
print(Counter(y_test))

Counter({np.int64(0): 2700, np.int64(1): 664})
Counter({np.int64(0): 696, np.int64(1): 146})


In [16]:
print(X_train)

[[[[ 24  19  49]
   [  3   0  24]
   [  7   4  23]
   ...
   [184  61 127]
   [173  48 115]
   [110  29 135]]

  [[ 31  25  60]
   [  6   2  30]
   [ 15  12  34]
   ...
   [178  55 122]
   [174  51 117]
   [105  26 131]]

  [[ 29  22  64]
   [  6   1  36]
   [ 13   8  37]
   ...
   [175  54 123]
   [173  52 117]
   [104  27 128]]

  ...

  [[ 53  19 101]
   [ 39   3  86]
   [ 50  15  96]
   ...
   [ 60  18 105]
   [ 62  24 107]
   [  3   0  15]]

  [[ 22   8  48]
   [ 18   4  44]
   [ 23   9  47]
   ...
   [ 26   9  45]
   [ 26  12  44]
   [  3   0   5]]

  [[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   2]
   [  0   0   2]
   [  0   0   2]]]


 [[[ 49  22  45]
   [ 63  23  68]
   [ 63  18  75]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  [[ 85  43  80]
   [107  41 110]
   [104  29 115]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  [[ 98  39  92]
   [126  40 125]
   [132  35 137]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0

In [17]:
from PIL import Image
IMAGE_DIR = "/home/alien/Git/DATA/mel_spects_wordrep/"

def get_image_dimensions(image_path):
    with Image.open(image_path) as img:
        width, height = img.size
    return width, height

image_path = '/home/alien/Git/DATA/mel_spects_wordrep/HeStutters_0_0_fluent.jpg'  # Change this to the path of your image file
width, height = get_image_dimensions(image_path)
print("Image width:", width)
print("Image height:", height)

Image width: 610
Image height: 450


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, MultiHeadAttention
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dropout, Flatten, Dense, Input, AveragePooling2D, Attention, Reshape, TimeDistributed, Bidirectional, LSTM, GRU
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input, decode_predictions
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.constraints import ClipValue
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Layer, MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization
from tensorflow.keras.layers import Input, ConvLSTM2D, MaxPooling2D, BatchNormalization, TimeDistributed, Flatten, Bidirectional, LSTM, Attention, Dropout, Dense

In [19]:
# def build_model(input_shape=(224, 224, 3)):
#     base_model = VGG19(weights='imagenet', include_top=True, input_tensor=Input(shape=input_shape))
#     # Get the output of the 'fc2' layer in VGG16
#     a = base_model.get_layer('fc2').output

#     # Add a Dense layer with 13 neurons
#     dense_layer = Dense(13, activation='relu')(a)

#     # Add a final output layer with sigmoid activation
#     output_layer = Dense(1, activation='sigmoid')(dense_layer)

#     # Define the model with VGG16 base and the added layers
#     model = Model(inputs=base_model.input, outputs=output_layer)

#     # Freeze the weights of the VGG16 layers
#     for layer in base_model.layers:
#         layer.trainable = False

#     return model

def build_model(input_shape=(224, 224, 3), use_float16=False):
    inputs = Input(shape=input_shape)

    # Convert to lower precision if specified
    if use_float16:
        x = tf.keras.layers.Lambda(lambda t: tf.cast(t, tf.float16))(inputs)
    else:
        x = inputs

    x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)
    
    x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)
    
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = TimeDistributed(Flatten())(x)  # Flatten along the time dimension
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Attention()([x, x])  # Self-attention mechanism

    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)  # Adding dropout with a rate of 0.5
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)  # Adding dropout with a rate of 0.5
    
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)

    return model

In [20]:
  # base_model = VGG19(weights='imagenet', include_top=True,
  #                   input_tensor=Input(shape=(224, 224, 3)))
  # base_model.summary()

In [21]:
vgg_model = build_model()
vgg_model.summary()

I0000 00:00:1734902364.868147   51527 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7196 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:13:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 112, 112,  │        256 │ max_pooling2d[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     73,856 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ max_pooling2d_1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │    295,168 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │      1,024 │ max_pooling2d_2[… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 28, 7168)  │          0 │ batch_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 28, 256)   │  7,472,128 │ time_distributed… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 28, 256)   │          0 │ bidirectional[0]… │
│ (Attention)         │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 7168)      │          0 │ attention[0][0] 

 Total params: 12,355,649 (47.13 MB)

 Trainable params: 12,354,753 (47.13 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
vgg_model.compile(
  optimizer=Adam(0.001),
  loss='binary_crossentropy',
  metrics=['accuracy']
)

In [23]:
from datetime import datetime
# batch_size = 16
# history = vgg_model.fit(
#     X_train,
#     y_train,
#     batch_size=batch_size,
#     epochs=35,
#     callbacks=[
#         ReduceLROnPlateau(
#             monitor = 'accuracy',
#             factor = 0.2,
#             patience = 5,
#             verbose = 1,
#             min_lr = 0.0001
#         ),
#         EarlyStopping(
#             monitor = 'accuracy',
#             patience = 10,
#             verbose = 1,
#             restore_best_weights = True
#         )
#     ]
# )

# early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor='accuracy', factor=0.2, patience=5, verbose=1, min_lr=0.0001)
logdir="./logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

batch_size = 40
history = vgg_model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=60,
    callbacks=[
        ReduceLROnPlateau(
            monitor = 'accuracy',
            factor = 0.2,
            patience = 5,
            verbose = 1,
            min_lr = 0.0001
        ),
        tensorboard_callback
        # early_stop  
    ]
)

Epoch 1/60


I0000 00:00:1734902369.337905   51674 cuda_dnn.cc:529] Loaded cuDNN version 90300


85/85 ━━━━━━━━━━━━━━━━━━━━ 20s 132ms/step - accuracy: 0.7362 - loss: 0.6240 - learning_rate: 0.0010
Epoch 2/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 127ms/step - accuracy: 0.7936 - loss: 0.5384 - learning_rate: 0.0010
Epoch 3/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 127ms/step - accuracy: 0.7930 - loss: 0.5249 - learning_rate: 0.0010
Epoch 4/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 127ms/step - accuracy: 0.7953 - loss: 0.5052 - learning_rate: 0.0010
Epoch 5/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 128ms/step - accuracy: 0.8045 - loss: 0.5004 - learning_rate: 0.0010
Epoch 6/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 128ms/step - accuracy: 0.8121 - loss: 0.4734 - learning_rate: 0.0010
Epoch 7/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 128ms/step - accuracy: 0.8048 - loss: 0.4500 - learning_rate: 0.0010
Epoch 8/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 128ms/step - accuracy: 0.8023 - loss: 0.4232 - learning_rate: 0.0010
Epoch 9/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 128ms/step - accuracy: 0.8143 - loss: 0.3950 - learning_rate: 0.0010
Epoch 10/60


In [24]:
# predictions
vgg_pred = vgg_model.predict(X_test, batch_size=1)

vgg_pred = np.round(vgg_pred)

  # model evaluation
confusion = confusion_matrix(y_test, vgg_pred)
print(classification_report(y_test, vgg_pred))
print(confusion)

842/842 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
              precision    recall  f1-score   support

           0       0.90      0.96      0.93       696
           1       0.73      0.47      0.57       146

    accuracy                           0.88       842
   macro avg       0.81      0.71      0.75       842
weighted avg       0.87      0.88      0.87       842

[[671  25]
 [ 78  68]]


In [25]:
vgg_model.save('./model_wordrep_AUGMENT.keras', overwrite=True)